In [1]:
from datasets import load_dataset
from tqdm.notebook import tqdm
from transformers import AutoTokenizer
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()

True

In [3]:
model_name = "unsloth/Qwen2.5-Coder-7B-Instruct"
# model_name = "meta-llama/Llama-2-7b-hf"

# model = AutoModelForCausalLM.from_pretrained(model_name,
#                                             #  torch_dtype=torch.float16
#                                              )
tokenizer = AutoTokenizer.from_pretrained(model_name, model_max_length=4096
                                          # torch_dtype=torch.float16
                                          )

In [4]:
# dataset = load_dataset('json', data_files='PyranetStatisticData/dataset_all_with_cell_num_error_timeout.jsonl', split = "train")
dataset = load_dataset('bnadimi/PyraNet-Verilog', split = "train")

In [6]:
dataset = dataset.select(range(10))

In [16]:
dataset

Dataset({
    features: ['code', 'description', 'text', 'token_length', 'extract_error'],
    num_rows: 692238
})

In [5]:
prompt = "this is the prompt that you are trying to get the token length for" * 4000
tokenized = tokenizer(prompt, return_tensors="pt")
print(tokenized["input_ids"].shape[1])

Token indices sequence length is longer than the specified maximum sequence length for this model (56001 > 4096). Running this sequence through the model will result in indexing errors


56001


In [5]:
import json, re, tempfile, subprocess, os
import numpy as np
from module_extraction import module_extraction

def formatting_prompts_func(examples, **kwargs):
    text_att = "text"
    format_chat = False
    if "format_chat" in kwargs:
        format_chat = kwargs["format_chat"]
        if kwargs["format_chat"]:
            text_att = "messages"

    descriptions = examples["description"]
    
    codes = examples["code"]
    texts = []
    token_lengths = []
    extract_errors = []
    for description, code in zip(descriptions, codes):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        description = description.replace("\\\\", "\\")
        description = json.loads(description)['description']
        code = code.replace("\\\\", "\\")
        
        extract_error = False
        
        try:
            _, _, _, module_def_span, _ = module_extraction(code)
            module_defs = "\n".join([code[span[0]:span[2]] for span in module_def_span])
        except:
            extract_error = True
            module_defs = ""
        
        instruction = "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."
        iinput = description + """
# You generate the required Verilog code only.
# The Verilog code must contain one of keywords performing logic operations, such as \"always\", \"and\", \"assign\", \"not\", \"nand\", \"nor\", \"or\", \"xnor\", \"xor\", or \"display\".
# Do not provide any explanation or comment.""" + "\n"*2 + module_defs
        output = code
        chat = [
            {"role": "system", "content": instruction},
            {"role": "user", "content": iinput},
            {"role": "assistant", "content": output}]

        if format_chat:
            text = json.dumps(chat)
        else:
            text = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=False, ) #+ tokenizer.eos_token
            text = re.sub(r'\n\n+', '\n', text)
        seq = tokenizer(text, return_tensors="pt")
        
        token_lengths.append(seq["input_ids"].shape[1])
        texts.append(text)
        extract_errors.append(extract_error)
        
    ret_dict = { 
        text_att: texts,
        "token_length": token_lengths,
        "extract_error": extract_errors
    }
    if format_chat:
        ret_dict["format"] = ["chatml"] * len(examples["description"])
    return ret_dict

In [6]:
dataset = dataset.map(formatting_prompts_func, batched = True, num_proc=os.cpu_count(),
                      fn_kwargs={"format_chat": True},
                     )
# test_dataset_t = dataset.select(range(1)).map(formatting_prompts_func, batched = True, num_proc=os.cpu_count(), fn_kwargs={"format_chat": True})

Map (num_proc=112):   0%|          | 0/692238 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (5448 > 4096). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (5221 > 4096). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (5283 > 4096). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (12590 > 4096). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (4752 > 4096). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence len

In [10]:
import numpy as np
train_index = np.load('TrainDataset/train_index2_0_5.npy')

dataset = dataset.select(train_index)
dataset

Dataset({
    features: ['code', 'description', 'messages', 'token_length', 'extract_error', 'format'],
    num_rows: 198220
})

In [16]:
with open('TrainDataset/Pyranet_chat_lines.jsonl', 'w') as outfile:
    outfile.write('')

In [17]:
import glob, os
from module_extraction import module_extraction
import subprocess, tempfile, json, signal, os

# for prompt_dir in all_prompt_dir:
# for i in range(len(all_num_cell_file)):
def do_process(ii):
    out_dict = {
            "messages": json.loads(dataset["messages"][ii]),
            'format': dataset["format"][ii]
        }
    return out_dict

In [18]:
from multiprocessing.managers import BaseManager
from multiprocess import Pool
from tqdm.notebook import tqdm
import os

num_core = os.cpu_count() - 4
my_range = range(1)#(len(dataset))

with open('TrainDataset/Pyranet_chat_lines.jsonl', 'w') as outfile:
    with Pool(processes=num_core) as pool:
        for i in tqdm(iterable=pool.imap(do_process, my_range), total=len(my_range)):
            json.dump(i, outfile)
            outfile.write('\n')

  0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
import json

with open('TrainDataset/Pyranet_chat_lines.jsonl', 'w') as outfile:
    for ii in range(len(dataset)):
        
        out_dict = {
            "messages": json.loads(dataset["messages"][ii]),
            'format': dataset["format"][ii]
        }
    # for entry in data_list:
        json.dump(out_dict, outfile)
        outfile.write('\n')

In [11]:
dataset.select_columns(['messages', 'format']).to_json('TrainDataset/Pyranet_chat.jsonl')

Creating json from Arrow format:   0%|          | 0/199 [00:00<?, ?ba/s]

737130093

In [8]:
dataset['messages'][0]

'[{"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."}, {"role": "user", "content": "The Verilog code defines a module named `VCC` that outputs a constant high logic level (1). The output `V` is always set to 1.\\n# You generate the required Verilog code only.\\n# The Verilog code must contain one of keywords performing logic operations, such as \\"always\\", \\"and\\", \\"assign\\", \\"not\\", \\"nand\\", \\"nor\\", \\"or\\", \\"xnor\\", \\"xor\\", or \\"display\\".\\n# Do not provide any explanation or comment.\\n\\nmodule VCC (output V);"}, {"role": "assistant", "content": "module VCC (output V);\\n   assign V = 1\'b1;\\nendmodule"}]'

In [20]:
print(dataset['text'][0])

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
The Verilog code defines a module named `VCC` that outputs a constant high logic level (1). The output `V` is always set to 1.
# You generate the required Verilog code only.
# The Verilog code must contain one of keywords performing logic operations, such as "always", "and", "assign", "not", "nand", "nor", "or", "xnor", "xor", or "display".
# Do not provide any explanation or comment.
module VCC (output V);<|im_end|>
<|im_start|>assistant
module VCC (output V);
   assign V = 1'b1;
endmodule<|im_end|>



In [18]:
dataset["token_length"][0]

158

In [10]:
dataset.select_columns(['text', 'token_length', 'extract_error']).to_json("TrainDataset/Pyranet_text_ex_error.jsonl")

Creating json from Arrow format:   0%|          | 0/693 [00:00<?, ?ba/s]

3291065255

In [17]:
dataset.select_columns(['text']).to_json("TrainDataset/Pyranet_text_only.jsonl")
dataset.select_columns(['text'])

Creating json from Arrow format:   0%|          | 0/693 [00:00<?, ?ba/s]

Dataset({
    features: ['text'],
    num_rows: 692238
})

In [19]:
print(dataset["text"][6000])

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
This Verilog code defines a test bench module for a Nios II CPU controller, simulating the CPU's behavior by connecting inputs and outputs that handle instruction execution and data management. It includes mechanisms to filter write data, register status and control signals, manage exceptions and interrupts, and validate data integrity during simulation. Additionally, it monitors state changes and generates error messages for any invalid or indeterminate ('x') signal states that may occur during operation.
# You generate the required Verilog code only.
# The Verilog code must contain one of keywords performing logic operations, such as "always", "and", "assign", "not", "nand", "nor", "or", "xnor", "xor", or "display".
# Do not provide any explanation or comment.
module controller_nios_0_cpu_test_bench (
                                          // inputs:
                 

In [14]:
tokenizer.model_max_length

4096

In [21]:
dataset

Dataset({
    features: ['code', 'description', 'text', 'token_length', 'extract_error'],
    num_rows: 692238
})

# Verilog-Eval Formatting

In [8]:
import glob, os, json
from module_extraction import module_extraction

# all_prompt_dir = glob.glob('../ext/verilog-eval/dataset_spec-to-rtl/Prob*_prompt.txt')
all_prompt_dir = glob.glob('../ext/verilog-eval/dataset_code-complete-iccad2023/Prob*_prompt.txt')
# appended_rows = {
#     'description': [], 
#     'code': [], 
#     'module_def_span': []
# }
all_prompt_dir.sort()

appended_rows = {'description': [],
                'code': []
                }
for prompt_dir in all_prompt_dir:
    prompt_dirname = os.path.dirname(prompt_dir)
    prompt_basename = os.path.basename(prompt_dir)
    code_basename = prompt_basename.replace('_prompt.txt', '_ref.sv')
    
    description = None
    code = None
    with open(prompt_dir, 'r') as file1, open(f'{prompt_dirname}/{code_basename}', 'r') as file2:
        description = file1.read()
        code = file2.read()
    _, _, _, module_def_span, _ = module_extraction(code)
    # appended_rows['description'].append(json.dumps({'description': description}))
    # appended_rows['code'].append(code)
    # appended_rows['module_def_span'].append(module_def_span)
    
    # 
    # appended_rows.append({
    #     'description': json.dumps({'description': description}), 
    #     'code': code, 
    #     'module_def_span': module_def_span.tolist()
    # })
    appended_rows['description'].append(json.dumps({'description': description}))
    appended_rows['code'].append(code)
appended_rows

{'description': ['{"description": "\\nBuild a circuit that always outputs a LOW.\\n\\nmodule TopModule (\\n  output zero\\n);\\n\\n"}',
  '{"description": "\\nBuild a circuit with no inputs and one output. That output should always\\ndrive 0 (or logic low).\\n\\nmodule TopModule (\\n  output out\\n);\\n\\n"}',
  '{"description": "\\nBuild a circuit with no inputs and one output. That output should always\\ndrive 1 (or logic high).\\n\\nmodule TopModule (\\n  output one\\n);\\n\\n"}',
  '{"description": "\\nBuild a circuit that reverses the byte order of a 32-bit vector.\\n\\nmodule TopModule (\\n  input [31:0] in,\\n  output [31:0] out\\n);\\n\\n"}',
  '{"description": "\\nCreate a module that implements a NOT gate.\\n\\nmodule TopModule (\\n  input in,\\n  output out\\n);\\n\\n"}',
  '{"description": "\\nGiven an 8-bit input vector [7:0], reverse its bit ordering.\\n\\nmodule TopModule (\\n  input [7:0] in,\\n  output [7:0] out\\n);\\n\\n"}',
  '{"description": "\\nCreate a module wit

In [10]:
from datasets import Dataset

ve_dataset = Dataset.from_dict(appended_rows)
ve_dataset

Dataset({
    features: ['description', 'code'],
    num_rows: 156
})

In [11]:
ve_dataset = ve_dataset.map(formatting_prompts_func, batched = True, num_proc=os.cpu_count())
ve_dataset

Map (num_proc=112):   0%|          | 0/156 [00:00<?, ? examples/s]

Dataset({
    features: ['description', 'code', 'text', 'token_length', 'extract_error'],
    num_rows: 156
})

In [13]:
print(ve_dataset['text'][0])

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Build a circuit that always outputs a LOW.
module TopModule (
  output zero
);
# You generate the required Verilog code only.
# The Verilog code must contain one of keywords performing logic operations, such as "always", "and", "assign", "not", "nand", "nor", "or", "xnor", "xor", or "display".
# Do not provide any explanation or comment.
module RefModule (
  output zero
);<|im_end|>
<|im_start|>assistant
module RefModule (
  output zero
);
  assign zero = 1'b0;
endmodule
<|im_end|>



In [15]:
ve_dataset.select_columns(['text']).to_json("TrainDataset/VE_text_156.jsonl")

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

234767

#

In [29]:
import glob
training_idxs = glob.glob('TrainDataset/train_index2*.npy')
total_idxs = 0
total_len = 0
len_arr = []
for training_idx in training_idxs:
    print(training_idx)
    training_idx_arr = np.load(training_idx)
    filter_dataset = dataset.select(training_idx_arr)

    #
    token_length_arr = np.array(filter_dataset['token_length'])
    token_length_valid_arr =np.where(token_length_arr <= tokenizer.model_max_length)
    filter_dataset = filter_dataset.select(token_length_valid_arr[0])
    
    #
    ext_err_arr = np.array(filter_dataset['extract_error'])
    ext_valid_arr =np.where(ext_err_arr == False)
    filter_dataset = filter_dataset.select(ext_valid_arr[0])

    training_idx_dataset_path = training_idx.replace('.npy', '.jsonl')
    filter_dataset.select_columns(['text']).to_json(training_idx_dataset_path)

    total_len += len(filter_dataset)
    len_arr.append(len(filter_dataset))
    # print(filter_dataset)
print(total_len, len_arr)

TrainDataset/train_index2_0_100.npy


Creating json from Arrow format:   0%|          | 0/92 [00:00<?, ?ba/s]

TrainDataset/train_index2_101_1000.npy


Creating json from Arrow format:   0%|          | 0/12 [00:00<?, ?ba/s]

TrainDataset/train_index2_1001_10000.npy


...

TrainDataset/train_index2_10001_100000.npy


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

TrainDataset/train_index2_100001_1100000.npy


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

105261 [91803, 11456, 1587, 377, 38]


In [30]:
import glob, os
from module_extraction import module_extraction
import subprocess, tempfile, json, signal, os

def do_process(i):
    print(i)

In [31]:
from multiprocessing.managers import BaseManager
from multiprocess import Pool
from tqdm.notebook import tqdm
import os

num_core = os.cpu_count() - 4
my_range = [(i, i) for i in range(2)]

# synth_error_ex = set()
# synth_timeout_ex = set()
# all_total_num_cells = [None] * (len(my_range))
with Pool(processes=num_core) as pool:
    for i in tqdm(iterable=pool.imap_unordered(do_process, my_range), total=len(my_range)):
        pass

(0, 0)

  0%|          | 0/2 [00:00<?, ?it/s]

(1, 1)



In [32]:
dataset

Dataset({
    features: ['code', 'description', 'text', 'token_length', 'extract_error'],
    num_rows: 692238
})

In [33]:
import glob, os, json
from module_extraction import module_extraction

# all_prompt_dir = glob.glob('../ext/verilog-eval/dataset_spec-to-rtl/Prob*_prompt.txt')
all_prompt_dir = glob.glob('../ext/verilog-eval/dataset_code-complete-iccad2023/Prob*_prompt.txt')
# appended_rows = {
#     'description': [], 
#     'code': [], 
#     'module_def_span': []
# }
all_prompt_dir.sort()

appended_rows = []
for prompt_dir in all_prompt_dir:
    prompt_dirname = os.path.dirname(prompt_dir)
    prompt_basename = os.path.basename(prompt_dir)
    code_basename = prompt_basename.replace('_prompt.txt', '_ref.sv')
    
    description = None
    code = None
    with open(prompt_dir, 'r') as file1, open(f'{prompt_dirname}/{code_basename}', 'r') as file2:
        description = file1.read()
        code = file2.read()
    _, _, _, module_def_span, _ = module_extraction(code)
    # appended_rows['description'].append(json.dumps({'description': description}))
    # appended_rows['code'].append(code)
    # appended_rows['module_def_span'].append(module_def_span)
    appended_rows.append({
        'description': json.dumps({'description': description}), 
        'code': code, 
        'module_def_span': module_def_span.tolist()
    })
appended_rows[0]

{'description': '{"description": "\\nBuild a circuit that always outputs a LOW.\\n\\nmodule TopModule (\\n  output zero\\n);\\n\\n"}',
 'code': "\nmodule RefModule (\n  output zero\n);\n\n  assign zero = 1'b0;\n\nendmodule\n\n",
 'module_def_span': [[0, 8, 36, 62]]}

In [36]:
ve_dataset = dataset.select_columns(['code', 'description']).select(range(0))
ve_dataset

Dataset({
    features: ['code', 'description'],
    num_rows: 0
})

In [38]:
for i in appended_rows:
    ve_dataset = ve_dataset.add_item(i)
ve_dataset

Dataset({
    features: ['description', 'code', 'module_def_span'],
    num_rows: 156
})

In [39]:
ve_dataset = ve_dataset.map(formatting_prompts_func, batched = True, num_proc=os.cpu_count())
ve_dataset

Map (num_proc=112):   0%|          | 0/156 [00:00<?, ? examples/s]

Dataset({
    features: ['description', 'code', 'module_def_span', 'text', 'token_length', 'extract_error'],
    num_rows: 156
})

In [43]:
np.where(np.array(ve_dataset['extract_error']) == True)

(array([], dtype=int64),)

In [40]:
ve_dataset['text']

'<｜begin▁of▁sentence｜>You are an AI programming assistant, utilizing the DeepSeek Coder model, developed by DeepSeek Company, and you only answer questions related to computer science. For politically sensitive questions, security and privacy issues, and other non-computer science questions, you will refuse to answer.### Instruction:\nBuild a circuit that always outputs a LOW.\nmodule TopModule (\n  output zero\n);\n# You generate the required Verilog code only.\n# The Verilog code must contain one of keywords performing logic operations, such as "always", "and", "assign", "not", "nand", "nor", "or", "xnor", "xor", or "display".\n# Further explanation must be construct as the syntax of Verilog comment.\nmodule RefModule (\n  output zero\n);\n### Response:\nmodule RefModule (\n  output zero\n);\n  assign zero = 1\'b0;\nendmodule\n<|EOT|>\n'

In [46]:
ve_dataset.select_columns(['text']).to_json('TrainDataset/VE_dataset.jsonl')

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

273143

# Timeout and Synthesis Error Explore

In [7]:
import glob

all_num_cell_file = glob.glob(".cache_count_num_cell_2/*")

In [8]:
import numpy as np

all_cell_num_with_no_null = np.array([[None, None]] * len(dataset))
all_cell_num_with_no_null

array([[None, None],
       [None, None],
       [None, None],
       ...,
       [None, None],
       [None, None],
       [None, None]], shape=(692238, 2), dtype=object)

In [9]:
import glob, os
from module_extraction import module_extraction
import subprocess, tempfile, json, signal, os

# for prompt_dir in all_prompt_dir:
# for i in range(len(all_num_cell_file)):
def do_process(i):
    filename_path = all_num_cell_file[i]
    file_name = os.path.basename(filename_path)
    logic_index = int(file_name.replace(".txt", ""))
    # print(file_name)
    with open(filename_path, 'r') as file:
        file_list = file.read().split(",")
        return (logic_index, tuple(file_list))
        # all_cell_num_with_no_null[logic_index][0] = int(file_list[0])
        # all_cell_num_with_no_null[logic_index][1] = int(file_list[1]) if file_list[1] != 'None' else None

In [10]:
from multiprocessing.managers import BaseManager
from multiprocess import Pool
from tqdm.notebook import tqdm
import os

num_core = os.cpu_count() - 4
my_range = range(len(all_num_cell_file))

# synth_error_ex = set()
# synth_timeout_ex = set()
# all_total_num_cells = [None] * (len(my_range))
with Pool(processes=num_core) as pool:
    for i in tqdm(iterable=pool.imap_unordered(do_process, my_range), total=len(my_range)):
        logic_index, file_list = i
        all_cell_num_with_no_null[logic_index][0] = int(file_list[0])
        all_cell_num_with_no_null[logic_index][1] = int(file_list[1]) if file_list[1] != 'None' else None

  0%|          | 0/692238 [00:00<?, ?it/s]

In [11]:
all_cell_num_with_no_null

array([[0, 0],
       [0, 2],
       [0, 1],
       ...,
       [0, None],
       [0, None],
       [0, None]], shape=(692238, 2), dtype=object)

In [16]:
unsynthesized_idxs = np.where(all_cell_num_with_no_null[:, 1] == None)[0]
unsynthesized_idxs

array([    23,     52,    144, ..., 692235, 692236, 692237],
      shape=(364709,))

In [17]:
all_cell_num_with_no_null = all_cell_num_with_no_null[unsynthesized_idxs]
all_cell_num_with_no_null

array([[0, None],
       [0, None],
       [0, None],
       ...,
       [0, None],
       [0, None],
       [0, None]], shape=(364709, 2), dtype=object)

In [20]:
synthesis_timout_idxs = np.where(all_cell_num_with_no_null[:, 0] == 1)[0]
synthesis_timout_idxs

array([   623,    624,    677, ..., 106693, 106695, 106698], shape=(2550,))

In [21]:
synthesis_error_idxs = np.where(all_cell_num_with_no_null[:, 0] == 0)[0]
synthesis_error_idxs

array([     0,      1,      2, ..., 364706, 364707, 364708],
      shape=(362159,))

In [26]:
dataset_timeout = dataset.select(synthesis_timout_idxs).select_columns(['token_length'])
dataset_timeout

Dataset({
    features: ['token_length'],
    num_rows: 2550
})

In [27]:
np.sum(np.array(dataset_timeout['token_length']))

np.int64(2863307)

In [28]:
dataset_error = dataset.select(synthesis_error_idxs).select_columns(['token_length'])
dataset_error

Dataset({
    features: ['token_length'],
    num_rows: 362159
})

In [29]:
np.sum(np.array(dataset_error['token_length']))

np.int64(466095468)